In [1]:
# ============================================================
# AI Route Planner in Logistics by Dr. Arpit Yadav
# Goal-Based Agent using LangGraph + Groq + Serper + Gradio
# ============================================================

# =========================
# 1. Install dependencies
# =========================
!pip -q install langgraph langchain langchain-groq gradio requests pandas pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.0 MB/s eta 0:00:00


In [2]:


# =========================
# 2. Import all necessary libraries
# =========================
import os
import json
import math
import time
import requests
import pandas as pd
from typing import TypedDict, List, Dict, Any

import gradio as gr

from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq

In [3]:



# =========================
# 3. Give Groq API and Serper API
# =========================
# You can set manually:
# os.environ["GROQ_API_KEY"] = "your_groq_api_key"
# os.environ["SERPER_API_KEY"] = "your_serper_api_key"

if "GROQ_API_KEY" not in os.environ:
    try:
        from getpass import getpass
        os.environ["GROQ_API_KEY"] = getpass("Enter GROQ_API_KEY: ")
    except:
        pass

if "SERPER_API_KEY" not in os.environ:
    try:
        from getpass import getpass
        os.environ["SERPER_API_KEY"] = getpass("Enter SERPER_API_KEY: ")
    except:
        pass

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
SERPER_API_KEY = os.getenv("SERPER_API_KEY", "")

Enter GROQ_API_KEY: ··········
Enter SERPER_API_KEY: ··········


In [4]:

# =========================
# 4. Select Model from Groq : use llama-3.1-8b-instant
# =========================
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [5]:
# =========================
# 5. Sample route database
# =========================
# We simulate route options for logistics planning.
# Goal-Based Agent chooses the route based on the user's goal.

SAMPLE_ROUTES = [
    {
        "route_id": "R1",
        "origin": "Bangalore",
        "destination": "Chennai",
        "distance_km": 350,
        "travel_time_hr": 7.0,
        "toll_cost": 1200,
        "fuel_cost": 2800,
        "risk_level": 2,
        "weather_impact": 1,
        "road_quality": 4
    },
    {
        "route_id": "R2",
        "origin": "Bangalore",
        "destination": "Chennai",
        "distance_km": 380,
        "travel_time_hr": 6.5,
        "toll_cost": 1800,
        "fuel_cost": 3000,
        "risk_level": 3,
        "weather_impact": 1,
        "road_quality": 5
    },
    {
        "route_id": "R3",
        "origin": "Bangalore",
        "destination": "Chennai",
        "distance_km": 330,
        "travel_time_hr": 8.0,
        "toll_cost": 900,
        "fuel_cost": 2600,
        "risk_level": 4,
        "weather_impact": 2,
        "road_quality": 3
    },
    {
        "route_id": "R4",
        "origin": "Mumbai",
        "destination": "Pune",
        "distance_km": 150,
        "travel_time_hr": 3.5,
        "toll_cost": 450,
        "fuel_cost": 1200,
        "risk_level": 2,
        "weather_impact": 1,
        "road_quality": 5
    },
    {
        "route_id": "R5",
        "origin": "Mumbai",
        "destination": "Pune",
        "distance_km": 165,
        "travel_time_hr": 3.0,
        "toll_cost": 650,
        "fuel_cost": 1300,
        "risk_level": 3,
        "weather_impact": 2,
        "road_quality": 4
    },
    {
        "route_id": "R6",
        "origin": "Delhi",
        "destination": "Jaipur",
        "distance_km": 280,
        "travel_time_hr": 5.0,
        "toll_cost": 1000,
        "fuel_cost": 2200,
        "risk_level": 2,
        "weather_impact": 2,
        "road_quality": 4
    },
    {
        "route_id": "R7",
        "origin": "Delhi",
        "destination": "Jaipur",
        "distance_km": 300,
        "travel_time_hr": 4.5,
        "toll_cost": 1300,
        "fuel_cost": 2400,
        "risk_level": 3,
        "weather_impact": 1,
        "road_quality": 5
    }
]


In [6]:
# =========================
# 6. LangGraph state
# =========================
class RouteState(TypedDict, total=False):
    origin: str
    destination: str
    cargo_type: str
    vehicle_type: str
    planning_goal: str
    search_mode: str
    budget_limit: float
    max_time_limit: float
    fuel_price_factor: float
    available_routes: List[Dict[str, Any]]
    web_evidence: str
    scored_routes: List[Dict[str, Any]]
    selected_route: Dict[str, Any]
    explanation: str
    final_report: str

In [7]:

# =========================
# 7. Helper functions
# =========================
def total_cost(route, fuel_price_factor=1.0):
    return route["toll_cost"] + route["fuel_cost"] * fuel_price_factor

def normalize(values):
    min_v = min(values)
    max_v = max(values)
    if max_v == min_v:
        return [0.5] * len(values)
    return [(v - min_v) / (max_v - min_v) for v in values]

def serper_search(query: str, num_results: int = 3) -> str:
    if not SERPER_API_KEY:
        return "Serper API key not available."

    url = "https://google.serper.dev/search"
    payload = json.dumps({"q": query, "num": num_results})
    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    try:
        response = requests.post(url, headers=headers, data=payload, timeout=20)
        response.raise_for_status()
        data = response.json()

        snippets = []
        for item in data.get("organic", [])[:num_results]:
            title = item.get("title", "")
            snippet = item.get("snippet", "")
            link = item.get("link", "")
            snippets.append(f"- {title} | {link}\n  {snippet}")

        if not snippets:
            return "No notable external logistics evidence found."
        return "\n".join(snippets)

    except Exception as e:
        return f"Serper search failed: {str(e)}"

In [8]:
# =========================
# 8. Goal-Based Agent logic
# =========================
# The agent selects actions/routes based on goal satisfaction.

def fetch_routes_node(state: RouteState) -> RouteState:
    origin = state.get("origin", "").strip()
    destination = state.get("destination", "").strip()

    filtered = [
        r for r in SAMPLE_ROUTES
        if r["origin"].lower() == origin.lower() and r["destination"].lower() == destination.lower()
    ]

    return {
        **state,
        "available_routes": filtered
    }


def optional_web_search_node(state: RouteState) -> RouteState:
    search_mode = state.get("search_mode", "Auto")
    origin = state.get("origin", "")
    destination = state.get("destination", "")

    if search_mode == "Off":
        return {**state, "web_evidence": "Web search disabled."}

    query = f"logistics road conditions traffic weather between {origin} and {destination}"

    if search_mode in ["On", "Auto"]:
        evidence = serper_search(query)
        return {**state, "web_evidence": evidence}

    return {**state, "web_evidence": "No web evidence used."}


def goal_based_planner_node(state: RouteState) -> RouteState:
    routes = state.get("available_routes", [])
    planning_goal = state.get("planning_goal", "Minimize Delivery Time")
    budget_limit = float(state.get("budget_limit", 100000))
    max_time_limit = float(state.get("max_time_limit", 100000))
    fuel_price_factor = float(state.get("fuel_price_factor", 1.0))

    if not routes:
        return {
            **state,
            "scored_routes": [],
            "selected_route": {}
        }

    times = [r["travel_time_hr"] for r in routes]
    distances = [r["distance_km"] for r in routes]
    costs = [total_cost(r, fuel_price_factor) for r in routes]
    risks = [r["risk_level"] + r["weather_impact"] for r in routes]
    road_qualities = [r["road_quality"] for r in routes]

    norm_time = normalize(times)
    norm_distance = normalize(distances)
    norm_cost = normalize(costs)
    norm_risk = normalize(risks)
    norm_road_quality = normalize(road_qualities)

    scored_routes = []

    for i, r in enumerate(routes):
        route_total_cost = total_cost(r, fuel_price_factor)

        budget_penalty = 0.25 if route_total_cost > budget_limit else 0
        time_penalty = 0.25 if r["travel_time_hr"] > max_time_limit else 0

        if planning_goal == "Minimize Delivery Time":
            score = (
                (1 - norm_time[i]) * 0.50 +
                (1 - norm_risk[i]) * 0.20 +
                (1 - norm_cost[i]) * 0.10 +
                (norm_road_quality[i]) * 0.20
            ) - budget_penalty - time_penalty

        elif planning_goal == "Minimize Cost":
            score = (
                (1 - norm_cost[i]) * 0.50 +
                (1 - norm_time[i]) * 0.15 +
                (1 - norm_risk[i]) * 0.20 +
                (norm_road_quality[i]) * 0.15
            ) - budget_penalty - time_penalty

        elif planning_goal == "Minimize Risk":
            score = (
                (1 - norm_risk[i]) * 0.50 +
                (norm_road_quality[i]) * 0.20 +
                (1 - norm_time[i]) * 0.15 +
                (1 - norm_cost[i]) * 0.15
            ) - budget_penalty - time_penalty

        elif planning_goal == "Balanced Optimization":
            score = (
                (1 - norm_time[i]) * 0.25 +
                (1 - norm_cost[i]) * 0.25 +
                (1 - norm_risk[i]) * 0.25 +
                (norm_road_quality[i]) * 0.25
            ) - budget_penalty - time_penalty

        else:
            score = (
                (1 - norm_time[i]) * 0.25 +
                (1 - norm_cost[i]) * 0.25 +
                (1 - norm_risk[i]) * 0.25 +
                (norm_road_quality[i]) * 0.25
            ) - budget_penalty - time_penalty

        cargo_adjustment = 0
        cargo_type = state.get("cargo_type", "General Goods")

        if cargo_type == "Perishable":
            cargo_adjustment += (1 - norm_time[i]) * 0.10
        elif cargo_type == "Fragile":
            cargo_adjustment += (1 - norm_risk[i]) * 0.10
        elif cargo_type == "High Value":
            cargo_adjustment += (1 - norm_risk[i]) * 0.12

        final_score = round(score + cargo_adjustment, 4)

        scored_routes.append({
            **r,
            "route_total_cost": round(route_total_cost, 2),
            "goal_score": final_score
        })

    scored_routes = sorted(scored_routes, key=lambda x: x["goal_score"], reverse=True)
    selected_route = scored_routes[0]

    return {
        **state,
        "scored_routes": scored_routes,
        "selected_route": selected_route
    }


def llm_explainer_node(state: RouteState) -> RouteState:
    selected_route = state.get("selected_route", {})
    scored_routes = state.get("scored_routes", [])
    planning_goal = state.get("planning_goal", "")
    cargo_type = state.get("cargo_type", "")
    vehicle_type = state.get("vehicle_type", "")
    web_evidence = state.get("web_evidence", "")

    if not selected_route:
        return {
            **state,
            "explanation": "No valid route found for the selected origin and destination."
        }

    top_routes_summary = "\n".join([
        f"- {r['route_id']}: time={r['travel_time_hr']} hr, cost={r['route_total_cost']}, risk={r['risk_level']}, road_quality={r['road_quality']}, score={r['goal_score']}"
        for r in scored_routes[:3]
    ])

    prompt = f"""
You are a logistics AI planning assistant.

Do not choose route. The route is already selected by a goal-based planner.
Your task is to explain why the selected route best matches the planning goal.

Inputs:
Planning Goal: {planning_goal}
Cargo Type: {cargo_type}
Vehicle Type: {vehicle_type}

Selected Route:
{json.dumps(selected_route, indent=2)}

Top Route Comparison:
{top_routes_summary}

External Web Evidence:
{web_evidence}

Write:
1. Executive summary
2. Why this route fits the goal
3. Route trade-offs
4. Final recommendation in one line

Keep it concise, professional, and business-ready.
"""

    try:
        resp = llm.invoke(prompt)
        explanation = resp.content if hasattr(resp, "content") else str(resp)
    except Exception as e:
        explanation = f"LLM explanation unavailable: {str(e)}"

    return {
        **state,
        "explanation": explanation
    }


def formatter_node(state: RouteState) -> RouteState:
    selected = state.get("selected_route", {})
    scored = state.get("scored_routes", [])
    explanation = state.get("explanation", "")
    web_evidence = state.get("web_evidence", "")

    if not selected:
        report = """
# AI Route Planner in Logistics

## Result
No route could be identified for the selected origin and destination.
        """.strip()
        return {**state, "final_report": report}

    ranking_lines = []
    for idx, r in enumerate(scored, start=1):
        ranking_lines.append(
            f"{idx}. {r['route_id']} | Distance: {r['distance_km']} km | Time: {r['travel_time_hr']} hr | "
            f"Cost: {r['route_total_cost']} | Risk: {r['risk_level']} | Score: {r['goal_score']}"
        )

    report = f"""
# AI Route Planner in Logistics

## Recommended Route
**Route ID:** {selected.get('route_id')}
**Origin:** {selected.get('origin')}
**Destination:** {selected.get('destination')}
**Distance:** {selected.get('distance_km')} km
**Travel Time:** {selected.get('travel_time_hr')} hr
**Total Cost:** {selected.get('route_total_cost')}
**Risk Level:** {selected.get('risk_level')}
**Road Quality:** {selected.get('road_quality')}
**Goal Score:** {selected.get('goal_score')}

## Route Ranking
{chr(10).join([f"- {line}" for line in ranking_lines])}

## Web Evidence
{web_evidence}

## Explanation
{explanation}
    """.strip()

    return {
        **state,
        "final_report": report
    }

In [9]:
# =========================
# 9. Build LangGraph
# =========================
builder = StateGraph(RouteState)

builder.add_node("fetch_routes", fetch_routes_node)
builder.add_node("optional_web_search", optional_web_search_node)
builder.add_node("goal_based_planner", goal_based_planner_node)
builder.add_node("llm_explainer", llm_explainer_node)
builder.add_node("formatter", formatter_node)

builder.set_entry_point("fetch_routes")
builder.add_edge("fetch_routes", "optional_web_search")
builder.add_edge("optional_web_search", "goal_based_planner")
builder.add_edge("goal_based_planner", "llm_explainer")
builder.add_edge("llm_explainer", "formatter")
builder.add_edge("formatter", END)

route_graph = builder.compile()

In [10]:

# =========================
# 10. Main function
# =========================
def run_route_planner(
    origin,
    destination,
    cargo_type,
    vehicle_type,
    planning_goal,
    budget_limit,
    max_time_limit,
    fuel_price_factor,
    search_mode
):
    state = {
        "origin": origin,
        "destination": destination,
        "cargo_type": cargo_type,
        "vehicle_type": vehicle_type,
        "planning_goal": planning_goal,
        "budget_limit": float(budget_limit),
        "max_time_limit": float(max_time_limit),
        "fuel_price_factor": float(fuel_price_factor),
        "search_mode": search_mode
    }

    result = route_graph.invoke(state)
    selected = result.get("selected_route", {})
    scored_routes = result.get("scored_routes", [])
    final_report = result.get("final_report", "")

    if scored_routes:
        df = pd.DataFrame(scored_routes)[[
            "route_id", "origin", "destination", "distance_km",
            "travel_time_hr", "route_total_cost", "risk_level",
            "road_quality", "goal_score"
        ]]
    else:
        df = pd.DataFrame(columns=[
            "route_id", "origin", "destination", "distance_km",
            "travel_time_hr", "route_total_cost", "risk_level",
            "road_quality", "goal_score"
        ])

    best_route = selected.get("route_id", "N/A")
    best_score = selected.get("goal_score", 0)
    total_cost_value = selected.get("route_total_cost", 0)
    eta = selected.get("travel_time_hr", 0)

    return final_report, best_route, best_score, total_cost_value, eta, df

In [11]:

# =========================
# 11. Example presets
# =========================
EXAMPLES = {
    "Fast Delivery: Bangalore to Chennai": (
        "Bangalore", "Chennai", "Perishable", "Truck",
        "Minimize Delivery Time", 6000, 10, 1.0, "Auto"
    ),
    "Low Cost: Bangalore to Chennai": (
        "Bangalore", "Chennai", "General Goods", "Truck",
        "Minimize Cost", 4500, 12, 1.0, "Auto"
    ),
    "Low Risk: Mumbai to Pune": (
        "Mumbai", "Pune", "Fragile", "Van",
        "Minimize Risk", 2500, 6, 1.0, "Auto"
    ),
    "Balanced: Delhi to Jaipur": (
        "Delhi", "Jaipur", "High Value", "Truck",
        "Balanced Optimization", 5000, 8, 1.0, "Auto"
    )
}

def load_example(example_name):
    return EXAMPLES[example_name]

In [ ]:

# =========================
# 12. Beautiful professional UI
# =========================
CUSTOM_CSS = """
body {
    background: linear-gradient(135deg, #f7fbff, #eefbf5);
}

.gradio-container {
    max-width: 1450px !important;
    margin: auto;
}

.hero-wrap {
    border-radius: 18px;
    padding: 18px 22px;
    background: linear-gradient(90deg, #eef6ff, #f4fff6);
    border: 1px solid #d9e7ff;
    box-shadow: 0 8px 24px rgba(0,0,0,0.06);
    overflow: hidden;
}

.hero-title {
    font-size: 34px;
    font-weight: 800;
    color: #184e77;
    margin-bottom: 4px;
}

.hero-subtitle {
    font-size: 14px;
    color: #4d6b86;
    margin-bottom: 12px;
}

.marquee {
    width: 100%;
    overflow: hidden;
    white-space: nowrap;
    border-radius: 10px;
    background: linear-gradient(90deg, #dff6e8, #e6f0ff);
    border: 1px solid #cfe3ff;
    padding: 10px 0;
}

.marquee span {
    display: inline-block;
    padding-left: 100%;
    animation: marquee 18s linear infinite;
    color: #0b6e4f;
    font-weight: 700;
    font-size: 18px;
}

@keyframes marquee {
    0%   { transform: translateX(0%); }
    100% { transform: translateX(-100%); }
}

.soft-card {
    border-radius: 16px !important;
    border: 1px solid #dfe9f3 !important;
    box-shadow: 0 6px 18px rgba(0,0,0,0.05) !important;
}

.footer-note {
    color: #5b7083;
    font-size: 13px;
}
"""

theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="green",
    neutral_hue="slate"
)

with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="AI Route Planner in Logistics") as demo:

    gr.HTML("""
    <div class="hero-wrap">
        <div class="hero-title">AI Route Planner in Logistics</div>
        <div class="hero-subtitle">Goal-Based Agent using LangGraph + Groq + Serper</div>
        <div class="marquee">
            <span>AI Route Planner in Logistics by Dr. Arpit Yadav • AI Route Planner in Logistics by Dr. Arpit Yadav • AI Route Planner in Logistics by Dr. Arpit Yadav</span>
        </div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=5):
            gr.Markdown("### Logistics Planning Input")

            example_choice = gr.Dropdown(
                choices=list(EXAMPLES.keys()),
                value="Fast Delivery: Bangalore to Chennai",
                label="Load Example Scenario",
                interactive=True
            )

            with gr.Row():
                origin = gr.Dropdown(
                    choices=["Bangalore", "Chennai", "Mumbai", "Pune", "Delhi", "Jaipur"],
                    value="Bangalore",
                    label="Origin"
                )
                destination = gr.Dropdown(
                    choices=["Bangalore", "Chennai", "Mumbai", "Pune", "Delhi", "Jaipur"],
                    value="Chennai",
                    label="Destination"
                )

            with gr.Row():
                cargo_type = gr.Dropdown(
                    choices=["General Goods", "Perishable", "Fragile", "High Value"],
                    value="Perishable",
                    label="Cargo Type"
                )
                vehicle_type = gr.Dropdown(
                    choices=["Truck", "Mini Truck", "Van", "Container"],
                    value="Truck",
                    label="Vehicle Type"
                )

            with gr.Row():
                planning_goal = gr.Dropdown(
                    choices=[
                        "Minimize Delivery Time",
                        "Minimize Cost",
                        "Minimize Risk",
                        "Balanced Optimization"
                    ],
                    value="Minimize Delivery Time",
                    label="Planning Goal"
                )

                search_mode = gr.Dropdown(
                    choices=["Auto", "On", "Off"],
                    value="Auto",
                    label="Web Search Enrichment"
                )

            with gr.Row():
                budget_limit = gr.Number(value=6000, label="Budget Limit")
                max_time_limit = gr.Number(value=10, label="Max Time Limit (hr)")
                fuel_price_factor = gr.Number(value=1.0, label="Fuel Price Factor")

        with gr.Column(scale=2):
            gr.Markdown("### Controls")
            run_btn = gr.Button("Run AI Route Planner", variant="primary")
            clear_btn = gr.Button("Clear", variant="secondary")

            gr.Markdown("""
            **How this works**
            - Finds matching route options
            - Applies goal-based route scoring
            - Optionally enriches with web evidence
            - Uses Groq LLM for explanation
            """)

    with gr.Row():
        best_route = gr.Textbox(label="Best Route ID", elem_classes=["soft-card"])
        best_score = gr.Number(label="Goal Score", elem_classes=["soft-card"])
        total_cost_box = gr.Number(label="Total Cost", elem_classes=["soft-card"])
        eta_box = gr.Number(label="ETA (hr)", elem_classes=["soft-card"])

    with gr.Row():
        report = gr.Markdown(label="Planner Report")
        route_table = gr.Dataframe(label="Ranked Route Options", interactive=False)

    example_choice.change(
        fn=load_example,
        inputs=example_choice,
        outputs=[
            origin, destination, cargo_type, vehicle_type,
            planning_goal, budget_limit, max_time_limit,
            fuel_price_factor, search_mode
        ]
    )

    run_btn.click(
        fn=run_route_planner,
        inputs=[
            origin, destination, cargo_type, vehicle_type,
            planning_goal, budget_limit, max_time_limit,
            fuel_price_factor, search_mode
        ],
        outputs=[
            report, best_route, best_score, total_cost_box,
            eta_box, route_table
        ]
    )

    clear_btn.click(
        fn=lambda: (
            "", "N/A", 0, 0, 0,
            pd.DataFrame(columns=[
                "route_id", "origin", "destination", "distance_km",
                "travel_time_hr", "route_total_cost", "risk_level",
                "road_quality", "goal_score"
            ])
        ),
        inputs=[],
        outputs=[report, best_route, best_score, total_cost_box, eta_box, route_table]
    )

    gr.Markdown(
        "<div class='footer-note'>Professional product-style route planning demo • Goal-Based Agent • LangGraph workflow • Groq explanation layer • Serper enrichment</div>"
    )

demo.launch(debug=True, share=True)

/tmp/ipykernel_11007/3532149767.py:78: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="AI Route Planner in Logistics") as demo:
/tmp/ipykernel_11007/3532149767.py:78: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="AI Route Planner in Logistics") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3aa572fcf9cd17da53.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Why this is a proper Goal-Based Agent

This application follows goal-based agent behavior because the agent:

receives a clear objective
evaluates available route choices
compares them against that objective
selects the route that best satisfies the goal

Examples:

Minimize Delivery Time
Minimize Cost
Minimize Risk
Balanced Optimization

The decision is not reflex-based.

The agent is not reacting only to current input with fixed rules.
Instead, it is planning toward a target outcome, which is the essence of a goal-based agent.